In [1]:
import os
import pandas as pd
import re
from pathlib import Path

In [2]:
alpha_path = Path("../data/01_raw/ADNI/ADNIAlpha")
beta_path = Path("../data/01_raw/ADNI/ADNIBeta")

In [3]:
def parse_adni_directory(base_path, folder_name):
    """
    Traverses the ADNI directory to find DICOM files and extracts 
    Patient ID and scan date from the folder structure.
    """
    print(f"Currently in {folder_name}")
    data = []
    
    if not base_path.exists():
        print(f"Can't find directory: {base_path}")
        return pd.DataFrame()

    # Regex to identify patients
    ptid_pattern = re.compile(r'\d{3}_S_\d{4}')
    
    # Regex to identify dates
    date_pattern = re.compile(r'\d{4}-\d{2}-\d{2}')

    for root, dirs, files in os.walk(base_path):
        # We only care about leaf folders that contain .dcm files
        dcm_files = [f for f in files if f.endswith('.dcm')]
        
        if dcm_files:
            path_parts = Path(root).parts
            
            # Extract PTID
            ptid = "Unknown"
            for part in path_parts:
                if ptid_pattern.match(part):
                    ptid = part
                    break
            
            # Extract Date
            scan_date = "Unknown"
            for part in path_parts:
                match = date_pattern.search(part) 
                if match:
                    scan_date = match.group(0)
                    break
            
            data.append({
                'Folder': folder_name,
                'PTID': ptid,
                'Scan_Date': scan_date,
                'Image_Count': len(dcm_files),
                'Full_Path': root
            })

    df = pd.DataFrame(data)
    print(f"Found {len(df)} scan directories in {folder_name}")
    return df

In [4]:
df_alpha = parse_adni_directory(alpha_path, "ADNIAlpha")
df_beta = parse_adni_directory(beta_path, "ADNIBeta")

Currently in ADNIAlpha
Found 406 scan directories in ADNIAlpha
Currently in ADNIBeta
Found 520 scan directories in ADNIBeta


In [5]:
all_images_df = pd.concat([df_alpha, df_beta], ignore_index=True)

In [6]:
# Analyze patients
if not all_images_df.empty:
    unique_alpha_patients = set(df_alpha['PTID'].unique())
    unique_beta_patients = set(df_beta['PTID'].unique())
    
    overlap = unique_alpha_patients.intersection(unique_beta_patients)
    alpha_only = unique_alpha_patients - unique_beta_patients
    beta_only = unique_beta_patients - unique_alpha_patients

    print("We are looking at patients descriptions per folder: ")
    print(f"Total scans found: {len(all_images_df)}")
    print(f"Total unique patients: {all_images_df['PTID'].nunique()}")
    print(f"Patients in ADNIAlpha: {len(unique_alpha_patients)}")
    print(f"Patients in ADNIBeta: {len(unique_beta_patients)}")
    print(f"Overlapping patients: {len(overlap)}")
    print(f"Alpha only patients: {len(alpha_only)}")
    print(f"Beta only patients: {len(beta_only)}")

    # Display sample data for verification
    print("Sample Data extracted:")
    print(all_images_df[['Folder', 'PTID', 'Scan_Date', 'Image_Count']].head())
    
    # all_images_df.to_csv("adni_image_inventory.csv", index=False)
else:
    print("No data found")

We are looking at patients descriptions per folder: 
Total scans found: 926
Total unique patients: 581
Patients in ADNIAlpha: 333
Patients in ADNIBeta: 333
Overlapping patients: 85
Alpha only patients: 248
Beta only patients: 248
Sample Data extracted:
      Folder        PTID   Scan_Date  Image_Count
0  ADNIAlpha  003_S_6014  2024-04-10          208
1  ADNIAlpha  130_S_4352  2017-07-03          104
2  ADNIAlpha  130_S_4352  2017-07-03          104
3  ADNIAlpha  130_S_4352  2017-07-03          211
4  ADNIAlpha  082_S_6564  2025-06-02          208
